# Test API Endpoint
Send a payload and see what comes back.


## Cell 1 — Health check

In [ ]:
import requests

BASE_URL = 'https://desrapidev-fqaphqeaeuc7hafh.westcentralus-01.azurewebsites.net'

r = requests.get(f'{BASE_URL}/', timeout=10)
print(f'Status: {r.status_code}')
print(r.json())


## Cell 2 — Send payload to /save

In [ ]:
import json, math
from pathlib import Path

JSON_FILE = './payloads/medicare_input_loadid142.json'

raw = json.loads(Path(JSON_FILE).read_text(encoding='utf-8'))

# Extract pbp rows — handles both {"pbp": [...]} and flat list
if isinstance(raw, list):
    rows = raw
elif isinstance(raw, dict) and 'pbp' in raw:
    rows = raw['pbp']
else:
    raise ValueError('Unexpected format')

# Sanitize NaN/Inf
def sanitize(obj):
    if isinstance(obj, dict):  return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)): return None
    return obj

payload = sanitize(rows)
print(f'Sending {len(payload):,} rows to {BASE_URL}/save')
print('Waiting for response (may take 1-3 min)...')

r = requests.post(
    f'{BASE_URL}/save',
    json=payload,
    headers={'Content-Type': 'application/json'},
    timeout=300,
)

print(f'\nHTTP Status: {r.status_code}')
print(json.dumps(r.json(), indent=2))


## Cell 3 — Send a tiny 3-row test payload
If Cell 2 gives a 500, run this first — it isolates whether the issue is the payload or the server.


In [ ]:
tiny_payload = [
    {
        'LoadID': 142,
        'FileName': 'Humana_H0028-014-000',
        'ID': 1069,
        'header': 'Rx',
        'category': 'Rx Setup',
        'field': 'Out-of-Network',
        'value': 'Yes',
        'DT': '9/20/2024 14:34'
    },
    {
        'LoadID': 142,
        'FileName': 'Humana_H0028-014-000',
        'ID': 1070,
        'header': 'Plan Characteristics',
        'category': 'Plan Details',
        'field': 'Plan Type',
        'value': 'HMO',
        'DT': '9/20/2024 14:34'
    },
    {
        'LoadID': 142,
        'FileName': 'Humana_H0028-014-000',
        'ID': 1071,
        'header': 'Benefit Details',
        'category': 'Primary Care Physician Services (7a) - Medicare',
        'field': 'Copayment amount',
        'value': '$0',
        'DT': '9/20/2024 14:34'
    },
]

print(f'Sending tiny test payload ({len(tiny_payload)} rows)...')
r = requests.post(
    f'{BASE_URL}/save',
    json=tiny_payload,
    headers={'Content-Type': 'application/json'},
    timeout=120,
)

print(f'HTTP Status: {r.status_code}')
print(json.dumps(r.json(), indent=2))


## Cell 4 — Retrieve results (run after Cell 2 succeeds)

In [ ]:
import pandas as pd

load_id = '142'  # change if different

r = requests.get(f'{BASE_URL}/results/{load_id}', timeout=30)
print(f'HTTP Status: {r.status_code}')

data = r.json()
print(f'Result count: {data.get("result_count", 0)}')

if data.get('results'):
    df = pd.DataFrame(data['results'])
    display(df[['benefitid','benefitname','serviceTypeDesc','tinyDescription']])
